In [ ]:
import json
from pathlib import Path

from rdflib import Graph
from rdfine import GraphReader
from compilers import PipelineGenerator, ProjectBuilder

In [ ]:
graph = Graph()

data_dir = Path("../data")
files = [
    "catalog.ttl",
    "pipeline_definition_nifi.ttl",
    "pipeline_definition_nifi.deployment.ttl",
    "tcs_shapes.ttl",
]
    
for filename in files:
    graph.parse(
        data_dir / filename,
        publicID="file:///workspace/pipeline/",
    )

reader = GraphReader(graph).infer(data_dir / "inference_rules.yaml")

In [ ]:
report = reader.validate(advanced=True, inference="rdfs")

violations = report.select(
    "?focus ?message",
    """
    ?result a sh:ValidationResult ;
        sh:focusNode ?focus ;
        sh:resultMessage ?message .
    """,
)

for row in violations.itertuples(index=False):
    print(f"Focus:   {row.focus}")
    print(f"Message: {row.message}")
    print()
    
if not report.ask("?report sh:conforms true"):
    for row in violations.itertuples(index=False):
        print(f"{row.focus}: {row.message}")

    raise ValueError("Pipeline definition does not conform")

In [ ]:
generator = PipelineGenerator(":DemonstratorPipeline", reader.graph)
build_graph = generator.compile()

[compiler.__name__ for compiler in generator.compilers]

In [ ]:
builder = ProjectBuilder(build_graph)

for _, file in builder.files.iterrows():
    content = file["content"]
    if file["filename"] == "flow.json":
        flow = json.loads(content)
        for kind in ("processors", "controllerServices"):
            for component in flow["rootGroup"][kind]:
                for name, descriptor in component["propertyDescriptors"].items():
                    if descriptor["sensitive"] and name in component["properties"]:
                        component["properties"][name] = "[REDACTED]"
        content = json.dumps(flow, indent=4)

    print(f"=== {file['filepath']}/{file['filename']} ===")
    print(content)

In [ ]:
written = builder.write("../out/nifi")

for path in written:
    print(path)